# Using the `tommos-adapter` in Ubermag

This notebook shows how to use the `tommos-adapter` for Ubermag.

In [ ]:
import micromagneticmodel as mm
import numpy as np
import pyvista as pv

import tommos_adapter as ta

## Generate mesh

We use `tommos` to generate the mesh:

In [ ]:
!tommos mesh --geom box --extent 20,20,20 --h 2 --out-name cube

## Initialize System

We start with a `System` object, with name `hysteresis`.

In [ ]:
system = mm.System(name="hysteresis")

## Define magnetization

We read the generated mesh and we define the following attributes.

- `m`: point-wise, defines the initial magnetization.
- `mat_id`: cell-wise, defines the material or region ID. Different IDs define separated regions rather than different materials, i.e. they can have the same magnetic properties. The previously executed program `tommos mesh` already defines this attribute.
- `Ms`: cell-wise, defines the spontaneous magnetization intrinsic magnetic property for each cell in A/m.
- `mesh_unit_length`: mandatory field-wise (understandable as mesh metadata) scalar argument, specifying the unit length in terms of meters.
- `mat_ids`: field-wise list of all possible material IDs. This parameter is only mandatory for multigrain meshes.
- `mat_names`: field-wise list of names of all material region. This parameter is only mandatory for multigrain meshes.

In [ ]:
mesh = pv.read("cube.vtu")

We define the initial magnetization uniformly as $(0, 1/\sqrt(2), 1/\sqrt(2))$:

In [ ]:
mesh.point_data["m"] = np.tile((0, 1 / np.sqrt(2), 1 / np.sqrt(2)), (mesh.n_points, 1))

We set the mesh unit length as 1e-9, as we consider the nanometer scale.

In [ ]:
mesh.field_data["mesh_unit_length"] = 1e-9

Add Spontaneous Polarization property as cell constant

In [ ]:
Js = 1.76  # spontaneous polarization in tesla
mesh.cell_data["Ms"] = np.full(mesh.n_cells, Js / mm.consts.mu0)

Finally, we define the material IDs and names.

In [ ]:
mesh.field_data["mat_ids"] = np.unique(mesh.cell_data["mat_id"])
mesh.field_data["mat_names"] = ["grain_1"]

In [ ]:
mesh

We can plot the glyph to show the uniform initial magnetization:

In [ ]:
mesh.glyph(orient="m", scale=False, tolerance=0.05).plot()

Finally, we assign the magnetization to `system.m`.

In [ ]:
system.m = mesh.copy()

## Define energy terms

Energy terms include intrinsic properties

In [ ]:
A = 1e-12  # exchange stiffness constant in J/m
K = 4e6  # first uniaxial anisotropy constant in J/m^3
u = (0, 0, 1)  # anisotropy unit axis
system.energy = mm.Exchange(A=A) + mm.UniaxialAnisotropy(K=K, u=u) + mm.Demag()
system.energy

## Define driver and hysteresis parameters

Next, we define the driver. The initialization does not need any argument.

In [ ]:
hd = ta.HysteresisDriver()
hd

Let us simulate hysteresis between $-2$ Tesla and $1$ Tesla applied along the $x$-direction in steps of $0.2$ Tesla:

In [ ]:
h_val = 5 / mm.consts.mu0
hd.drive(system, h_direction=(0, 0, 1), h_start=h_val, h_final=-h_val, n=30)

In [ ]:
system.m

In [ ]:
system.m.glyph(orient="m", scale=False, tolerance=0.05).plot()

## Run full hysteresis loop

First, we reset the system to a uniform magnetization.

In [ ]:
system.m = mesh.copy()

To run a full hysteresis loop, we specify `full_loop=True` when calling `.drive()`:

In [ ]:
h_val = 5 / mm.consts.mu0
hd.drive(system, h_direction=(0, 0, 1), h_start=h_val, h_final=-h_val, n=30, full_loop=True)

In [ ]:
system.table.mpl(y=["J_par"])